In [1]:
import pandas as pd

In [69]:
import numpy as np

_k = 2
_ret = 'e5'
_dataset = 'nq_dev'

experiment_name = f'{_ret}_{_dataset}'
retr_res = pd.read_csv(f'../../rag_utility/res/{experiment_name}.csv')
qualt5_df = pd.read_csv(f'./quality_res/{experiment_name}.csv')
qualt5_df.qid = qualt5_df.qid.astype('str')

qualt5_df = qualt5_df[qualt5_df['rank']<_k]
qualt5_df.quality = qualt5_df.quality.apply(lambda x: np.exp(x))

temp_df = qualt5_df.groupby(['qid']).quality.apply(lambda x: x.max())
max_score_dict = dict(zip(temp_df.index.tolist(), list(temp_df.values)))

temp_df = qualt5_df.groupby(['qid']).quality.apply(lambda x: x.mean())
avg_score_dict = dict(zip(temp_df.index.tolist(), list(temp_df.values)))

In [70]:
import json
import numpy as np

if('nq' in _dataset):
    _dataset_dev, _prefix, _suffix = _dataset, 'short', 'concise'
else:
    _dataset_dev, _prefix, _suffix = 'dev_small', 'random', 'prompt1'

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_1calls_0_0_bm25_dl_{_dataset_dev}_{_suffix}_eval.json')
zero_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}.json')
k_gens = json.load(f)
f.close()

f = open(f'../coherence_eval/log_prob_temp_res/full_context/{_dataset_dev}_{_ret}_{_k}.json')
perpC = json.load(f)
f.close()

dev_res = retr_res[['qid', 'query']].drop_duplicates().copy()
dev_res.qid = dev_res.qid.astype('str')
dev_res = dev_res[dev_res.qid.isin(qualt5_df.qid.unique())]
dev_res['avg_quality'] = dev_res.qid.apply(lambda x: avg_score_dict[x])
dev_res['max_quality'] = dev_res.qid.apply(lambda x: max_score_dict[x])

if('nq' in _dataset):
    base_f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items() if ('0' in item[1].keys())}
else:
    base_f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in k_evals.items() if ('0' in item[1].keys())}

kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

dev_res = dev_res[dev_res.qid.astype('str').isin(f1_dict.keys())]
dev_res['f1'] = dev_res.qid.apply(lambda x: f1_dict[str(x)])
dev_res['utility'] = dev_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])

dev_res = dev_res.dropna(axis='index')
dev_res.head(3)

,qid,query,avg_quality,max_quality,f1,utility
0,dev_0,who sings does he love me with reba,0.710262,0.735915,1.000000,1.000000
120,dev_1,how many pages is invisible man by ralph ellison,0.749081,0.762247,0.000000,0.000000
240,dev_2,where do the great lakes meet the ocean,0.611230,0.636881,0.857143,0.857143


In [71]:
from scipy import stats

print(stats.spearmanr(dev_res.avg_quality, dev_res.f1)[0], stats.kendalltau(dev_res.avg_quality, dev_res.f1)[0])
print(stats.spearmanr(dev_res.avg_quality, dev_res.utility)[0], stats.kendalltau(dev_res.avg_quality, dev_res.utility)[0])

0.06408798076874538 0.04837522024054099
0.04364281637742659 0.032197805359914636
